# ASE Interface

*Lucas Visscher, Vrije Universiteit Amsterdam, 2025*

Since release DIRAC25 we provide a Python interface to the Atomistic Simulation Environment that can be used to automate calculations.

A prerequisite to this interface is to follow the installation instructions in the `ase_dirac` folder. As a minimum you need to do (in your favourite python environment, the interface has been tested inside a conda environment), assuming you are in the DIRAC top directory:

1. ``cd build; export PATH=`pwd`:$PATH; cd ..`` (add's DIRAC's build directory to your PATH to make the `pam` command available)
2. `cd ase_dirac; pip install --user -e .; cd ..` (to install ase and the ase_dirac interface)

After this step you may automate DIRAC calculations. The next code cell checks whether the interface is installed correctly. If it gives an error, you should check whether there was an issue with any of the two required steps above.

In [ ]:
from ase import Atoms
from ase.io import read
from ase.build import molecule
from ase.units import Ha
from ase_dirac import DIRAC

As a first test we run an SCF calculation for the hydrogenfluoride molecule and print the energy.

In [ ]:
molecule_label = 'HF'
ase_molecule = molecule(molecule_label)
ase_molecule.calc = DIRAC(
              wave_function={'.scf': '',
                             '*scf': {'.ergcnv': '1.E-8 1.E-6',
                                      '.maxitr': '35'},
                            },
              molecule={'*basis': {'.default': 'cc-pVDZ'}},
              label=molecule_label)
print(f'The Hartree-Fock energy of {molecule_label} is ',ase_molecule.get_potential_energy() / Ha,' Hartree.')

In the next cell we run a more extensive calculation in which we tabulate MP2 energies computed with different Hamiltonians for three small molecules. In this piece of Python code we also allow for potential failures of a calculation, using the `try` and `except` mechanism of Python.

In [ ]:
# Some simple small molecules taken from the G2 set in the ASE database
molecules = ['H2O','N2','F2']
# Defines the **HAMILTONIAN input block, note the two lines of input for the full DCG Hamiltonian
hamiltonian_inputs = ['.nonrel','.x2cmmf','.gaunt\n.dossss']
# Labeling the different inputs with a descriptive string
calculation_labels = ['NR-MP2','X2Cmmf-MP2','DCG-MP2']
# Making an empty list to store results (energies) for each molecule
results = []

# Outermost loop over different molecules
for mol in molecules:
    ase_molecule = molecule(mol)
    energy = []
    # Loop over the three different Hamiltonians that we consider
    for ham, calc in zip(hamiltonian_inputs,calculation_labels):
        label=calc+'_'+mol
        try:
            ase_molecule.calc = DIRAC(
              hamiltonian={ham: ''},
              wave_function={'.scf': '', '.mp2': '',
                             '*scf': {'.ergcnv': '1.E-8 1.E-6',
                                      '.maxitr': '35'},
                             '*mp2cal': {'.occup': 'all','.virtual': 'all'},
                            },
              molecule={'*basis': {'.default': 'cc-pVDZ'}},
              label=label)
            energy.append(ase_molecule.get_potential_energy() / Ha)
        except:
            print (f'Calculation {label} could not be run correctly, please check')
            energy.append(0)
    results.append(energy)
    
# Print results in a table
print(' Møller-Plesset second order (MP2) energies computed with different Hamiltonians\n')
print(80*'-')
print('  {:<15} {:>20} {:>20} {:>20}'.format('Molecule',*calculation_labels))
print(80*'-')
for mol, result in zip(molecules,results):
    print ('  {:<15} {:>20.8f} {:>20.8f} {:>20.8f}'.format(mol,*result))
print(80*'-')

Note that the relativistic energies are always lower (more negative) than the non-relativistic ones, and the good approximation that the X2Cmmf method provides, with energies slightly below the DCG reference due to the neglect of the Gaunt interaction in the default X2Cmmf scheme.

# Input definition and calculator constructor methods
In the cells above, we defined the calculation setup using the internal dictionary representation of the `DIRAC` calculator object. This is handy in case you can import molecular structures from an ASE-compatible database and need to make only minimal changes to the the default DIRAC input. 

To be able to have more control over your calculations, you can also first write the required two input files of DIRAC. Below we do so by defining them inside the notebook as a string and then writing and reading these files. If you already have these inputs on file you can also point the notebook to the right files to read. 

##  Construction of the calculator setup
We like to construct an ASE `Atoms` object with a DIRAC calculator that is ready to interact with the functionalities of ASE. For this we define two files `F2.xyz` and `dirac_example.inp`:

In [ ]:
xyz_string = """2
F2
O      0 0 0
O      0.00000       -0.75545     -0.47116
"""

fscc_str = """ **DIRAC
    .TITLE
    Studying double ionization with FSCC.
    .WAVE F
    **HAMILTONIAN
    .LVCORR
    **WAVE FUNCTIONS
    .SCF
    .RELCCSD
    *SCF
    .PRINT 
    1
    **RELCCSD
    .ENERGY
    .PRINT
    1
    .FOCKSP
    *CCFSPC
    .DOIE2
    .NACTH
    1 1 0 0
    .MAXIT
    200
    **MOLECULE
    *BASIS
    .DEFAULT
    dyall.2zp
    *SYMMETRY
    .LINSYM
    *END OF
"""

with open("F2.xyz", "w") as f:
    f.write(xyz_string)

with open("dirac_example.inp", "w") as f:
    f.write(fscc_str)

We can now define an atoms object from the DIRAC input file and the xyz file, and run the calculations as in the previous part of the tutorial:

In [ ]:
molecule = DIRAC.ASE_atoms_from_inpfile_and_coords("dirac_example.inp", "F2.xyz")
molecule.get_potential_energy() / Ha

If you already have the input files you can in this way run the corresponding DIRAC calculation inside a notebook with only two lines of code. You can also attach the DIRAC calculator to a predefined `Atoms` object if this desired (for example if some previous interaction via ASE is required). For instance like done in the cell below that uses the xtb preoptimizer if this calculator is installed or sets the bond length explicitly in case it is not installed.

In [ ]:
from ase import Atoms

# ============================================================
# Try to optimize HCl using xTB (recommended pre-optimizer)
#
# Install instructions (one-time):
#   conda install -c conda-forge xtb xtb-python
#
# If xTB is not available, fall back to a reasonable
# fixed H–Cl geometry (1.27 Å).
# ============================================================

try:
    from xtb.ase.calculator import XTB
    from ase.optimize import BFGS

    print("Using xTB (GFN2-xTB) for geometry optimization.")

    # Start from a deliberately stretched bond
    atoms = Atoms(
        symbols=["H", "Cl"],
        positions=[
            (0.0, 0.0, 0.0),
            (2.5, 0.0, 0.0)
        ]
    )

    atoms.calc = XTB(
        method="GFN2-xTB",
        verbosity=0
    )

    opt = BFGS(atoms)
    opt.run(fmax=0.01)

except Exception as e:
    print("xTB not available — falling back to fixed HCl geometry.")
    print(f"(Reason: {type(e).__name__})")

    # Accepted experimental H–Cl bond length ≈ 1.27 Å
    atoms = Atoms(
        symbols=["H", "Cl"],
        positions=[
            (0.0, 0.0, 0.0),
            (1.27, 0.0, 0.0)
        ]
    )

# ------------------------------------------------------------
# Geometry printout (Option 1)
# ------------------------------------------------------------
symbols = atoms.get_chemical_symbols()
positions = atoms.get_positions()

print("\nHCl geometry:")
for s, (x, y, z) in zip(symbols, positions):
    print(f"{s:2s}  {x:10.6f}  {y:10.6f}  {z:10.6f}")

print(f"\nH–Cl bond length: {atoms.get_distance(0, 1):.4f} Å")

In [ ]:
hcl = DIRAC.ASE_atoms_from_inpfile("dirac_example.inp", atoms)
hcl.get_potential_energy() / Ha


---

## Calculator constructors
Running DIRAC in this way is very simple, but it is instructive to see what happens under the hood. ASE's `Atoms` requires a `calc` object to run calculations. In our case we use the custom `DIRAC` calculator object. Therefore, it is necessary to initialize this object and attach it to the `Atoms`. Once the xyz and input files are defined, it is possible to construct the object as:

```python
ase_molecule = read("molecule.xyz", format="xyz")
ase_molecule.calc = DIRAC.from_file("dirac_example.inp")
```

Or :

```python
ase_molecule.calc = DIRAC.from_string(fscc_str)
```

For instance for this calculation on O$_2$ that will be similar to the one of F$_2$, but starts from the dianion.

In [ ]:
O2_string="""2
O2 - geometry from NIST
O      0 0 0
O      0.0       0.0     1.20752
"""

with open("O2.xyz", "w") as f:
    f.write(xyz_string)

ase_molecule = read("O2.xyz", format="xyz")
ase_molecule.calc = DIRAC.calculator_from_file("dirac_example.inp")

# Alternatively (as already illustrated above) the calculator can be defined using a string as:
# ase_molecule.calc = DIRAC.calculator_from_string(fscc_str)

To check the input definition we can look at the internal dictionary definition that this calculator presents:

In [ ]:
param_list = [
    f"{key}, {ase_molecule.calc.parameters[key]}\n"
    for key in ase_molecule.calc.parameters.keys()
]

print("".join(param_list))

If for whatever reason it is necessary to modify some input parameters for a specific molecule, it is possible to start from a template `.inp` file and then modify the dictionary keyword-value pairs to override these defaults. We do this below, updating the charge as well as the number of active holes. The calculation then gives the states of the neutral molecule as they can be reached by double ionization from the di-anion.

In [ ]:
ase_molecule.calc = DIRAC.calculator_from_file("dirac_example.inp")

ase_molecule.calc.parameters['molecule'].update({'*charge': {'.charge': '-2'}})
ase_molecule.calc.parameters['wave_function'].update({'*ccfspc': {'.doie2': '', '.nacth': '2 2 0 0', '.maxit': '200'}})
param_list = [
    f"{key}, {ase_molecule.calc.parameters[key]}\n"
    for key in ase_molecule.calc.parameters.keys()
]

print("".join(param_list))

try:
    ase_molecule.calc.calculate()
except TypeError:
    ase_molecule.get_potential_energy() / Ha


---
# Calculation results and I/O
Once a calculation has finished, it is possible to interact with the results directly using the `DIRAC` calculator object. We can check some common data such as energies and eigenvalues direcly by calling:


In [ ]:
ase_molecule.calc.results

## h5 functionality from the calculator 
Under the hood what is happening is that the `read_results()` method in the calculator was extended to retrieve some data from the `CHECKPOINT.h5` file generated in the calculation: 

```python
def read_results(self): 
    """Read energy from DIRAC output file. Try to extract from h5 if availiable"""
    out_file = self.label + ".out"
    output = read_dirac_out(out_file)
    self.results = output
    self._extract_common_results_from_h5()
```

Where `_extract_common_results_from_h5()` extracts from the associated `.h5` file what now is an arbitrary list of (probably) values of interest: 

```python
        commonly_requested_results: dict = {
            "result/wavefunctions/scf/energy": "scf_energy",
            "result/wavefunctions/scf/mobasis/eigenvalues": 'scf_eigenvalues',
        }
```

A property `DIRAC.h5_contents` property was added to list all availiable properties in the `.h5` file: 

In [ ]:
ase_molecule.calc.h5_contents[0:10]

And a `.extract_h5_value` method was added in order to extract properties by key. We can for instance look at
the number of ao's in the large component basis:

In [ ]:
n_ao = ase_molecule.calc.extract_h5_value("input/aobasis/1/n_ao")
print(*n_ao)

For more information on how to read DIRAC's `.h5` files, check the `.h5` tutorials.